# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR^2 CRC Survivor dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library in Python.

### Dataset Source

The dataset source is provided via a Croissant schema URL. All references to record sets, fields, and columns use the respective `@id` identifiers.

In [ ]:
# Install mlcroissant if not already present
!pip install -q mlcroissant

## 1. Data Loading

We load the dataset metadata and records via the Croissant schema using `mlcroissant`. This makes both the metadata and the records accessible using unique `@id` references.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
# Optionally display dataset version, citation etc.
print(f"Dataset ID: {metadata.id}\nVersion: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview

Let's review the available record sets, fields, and their `@id`s. This helps us understand the underlying data structure for further processing and data extraction.

**List all Record Sets in the dataset:**

In [ ]:
# List all available record sets in the dataset
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the schema. This is likely a single main table dataset, using default record set.")

    # mlcroissant typically exposes a main table via detected columns/fields; let's enumerate those
    # Fetch the default (only) record set ID
    # For many flat datasets, it's just the dataset's own @id plus '/table' or similar
    inferred = False
    for r in dataset._backend._model_by_id.values():
        if getattr(r, '@type', None) == 'RecordSet':
            print(f"Found record set: {r['@id']}")
            record_sets.append(r['@id'])
            inferred = True
    if not inferred:
        # Fallback: try default naming convention
        record_sets = [metadata.id + "/table"]            
    print("Record Sets IDs:", record_sets)
else:
    for rs in record_sets:
        print(f"Record Set: {rs}")    

# For the rest of this notebook, we select the *first* detected record set (main table)
main_record_set_id = record_sets[0]

**List all Fields and corresponding `@id`s in the main record set:**

In [ ]:
# Inspect fields and columns in chosen record set
fields = dataset.fields(record_set=main_record_set_id)
print(f"Fields in record set {main_record_set_id}:")
for f in fields:
    # Each field has 'id', 'name', and possibly 'data_type'.
    print(f"- id: {f.id}  |  name: {getattr(f, 'name', None)}  |  data_type: {getattr(f, 'data_type', None)}")

**Preview the records (rows) using their field `@id`s:**

In [ ]:
# Print the first few records (each record is a dict keyed by field @id)
print(f"\nFirst two records in record set {main_record_set_id}:")
for i, row in enumerate(dataset.records(record_set=main_record_set_id)):
    print(row)
    if i >= 1:
        break

## 3. Data Extraction

We will load the data from the main record set into a Pandas DataFrame for analysis. All columns will be referenced using their field `@id`s.


In [ ]:
# Extract ALL records from the main record set into a DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded Data Columns (field @id):\n{list(df.columns)}")
display(df.head())

## 4. Exploratory Data Analysis (EDA)

Let's apply some common EDA steps:

- Select a numeric field (such as age or time interval in days, by `@id`)
- Filter records based on this field
- Normalize the numeric values
- Optionally, group by another field and compute means.

**First, identify possible numeric fields from the earlier field listing:**

In [ ]:
# Identify numeric fields by examining data types and previewing column values
numeric_field_candidates = []
sample_row = df.iloc[0]
for col in df.columns:
    if isinstance(sample_row[col], (int, float, np.integer, np.floating)):
        numeric_field_candidates.append(col)
    else:
        # Try to coerce to float to discover numeric-like fields (e.g. if some are strings)
        try:
            float(sample_row[col])
            numeric_field_candidates.append(col)
        except:
            pass
print(f"Numeric candidate fields (@id): {numeric_field_candidates}")

For this analysis, let's select the first detected numeric field `@id` (or, if known, the field corresponding to patient age at diagnosis or diagnosis interval).

We demonstrate filtering, normalization, and grouping on this field.

In [ ]:
# -- Set up fields for analysis, referencing by @id --
# If age or time fields are present, use those; else fallback to the first numeric candidate
# (Replace the example field IDs below with those printed from field list for your dataset)

# Example guess for field @id for age or time interval
# Uncomment & set manually as needed; here we take from earlier identified numeric candidates
import re

# Prioritize age or interval
field_priority_patterns = [r'age', r'interval', r'days', r'months', r'year']
selected_numeric_field = None
for pattern in field_priority_patterns:
    for field_id in numeric_field_candidates:
        if re.search(pattern, field_id, re.I):
            selected_numeric_field = field_id
            break
    if selected_numeric_field:
        break
if not selected_numeric_field and numeric_field_candidates:
    selected_numeric_field = numeric_field_candidates[0]
print(f"Chosen numeric field: {selected_numeric_field}")

# Clean up the column: coerce to numeric, handle missing.
df[selected_numeric_field] = pd.to_numeric(df[selected_numeric_field], errors='coerce')
threshold = df[selected_numeric_field].median()
filtered_df = df[df[selected_numeric_field] > threshold].copy()
print(f"Filtered records with {selected_numeric_field} > {threshold:.2f} (median): {len(filtered_df)} of {len(df)} records.")
display(filtered_df[[selected_numeric_field]].head())

# Normalize the field
filtered_df[f"{selected_numeric_field}_normalized"] = (
    (filtered_df[selected_numeric_field] - filtered_df[selected_numeric_field].mean()) /
    filtered_df[selected_numeric_field].std(ddof=0)
)
print(f"\nDistribution of normalized {selected_numeric_field}:")
display(filtered_df[[selected_numeric_field, f"{selected_numeric_field}_normalized"]].head())

# Try grouping by a categorical field (e.g., MSI status, sex, anatomical location, etc.).
# First list non-numeric fields
non_numeric_field_candidates = [col for col in df.columns if col not in numeric_field_candidates]
print(f"Non-numeric candidate fields (@id): {non_numeric_field_candidates}")

# Pick first non-numeric as group field (override here for known @id, e.g. MSI status @id)
group_field = None
priority_group_patterns = [r'msi', r'sex', r'location', r'subtype', r'group', r'stage', r'histology']
for pat in priority_group_patterns:
    for col in non_numeric_field_candidates:
        if re.search(pat, col, re.I):
            group_field = col
            break
    if group_field:
        break
if not group_field and non_numeric_field_candidates:
    group_field = non_numeric_field_candidates[0]
print(f"Grouping field: {group_field}")

if group_field:
    grouped_df = filtered_df.groupby(group_field)[selected_numeric_field].mean().to_frame()
    print(f"Grouped means of {selected_numeric_field} by {group_field}:")
    display(grouped_df.head())

## 5. Visualization

Let's make some basic charts! For demonstration, plot the selected numeric field distribution, and, if grouping is possible, a group-wise mean plot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 5))
sns.histplot(df[selected_numeric_field].dropna(), kde=True, bins=15, color='mediumseagreen')
plt.title(f"Distribution of {selected_numeric_field}")
plt.xlabel(selected_numeric_field)
plt.ylabel('Count')
plt.show()

# If grouped_df exists, plot group-wise means
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(8, 5))
    grouped_df_sorted = grouped_df.sort_values(by=selected_numeric_field, ascending=False)
    sns.barplot(x=grouped_df_sorted.index, y=selected_numeric_field, data=grouped_df_sorted, palette="viridis")
    plt.xticks(rotation=45, ha='right')
    plt.title(f"Mean {selected_numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {selected_numeric_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We have successfully loaded and explored the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id` fields.
- The data includes clinicopathological and molecular attributes for second primary colorectal cancer in cancer survivors.
- We identified and analyzed a key numeric field (e.g., age or interval) and grouped by a clinically meaningful field (e.g., MSI status).
- Visualizations highlighted data distributions and group-wise means. This pipeline enables further domain-specific analyses on the dataset.

_For more advanced workflows or customized analyses, refer to the official [Croissant documentation](https://mlcommons.github.io/croissant/python/)_